# 工具的使用 tools
> tool calling == function calling

## 1. @tool 装饰器：创建自定义工具

In [1]:
from langchain_core.tools import tool

# 使用 @tool 装饰器创建工具
# docstring 会自动成为工具的描述，LLM 根据描述决定是否调用此工具

@tool
def add(a: int, b: int) -> int:
    """两个整数相加，返回结果。"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """两个整数相乘，返回结果。"""
    return a * b

# 测试工具
print(f"add(3, 5) = {add.invoke({'a': 3, 'b': 5})}")
print(f"multiply(3, 5) = {multiply.invoke({'a': 3, 'b': 5})}")

# 查看工具的 schema（LLM 通过这个来理解如何调用）
print(f"\n工具名称：{add.name}")
print(f"工具描述：{add.description}")
print(f"参数 schema：{add.args}")

add(3, 5) = 8
multiply(3, 5) = 15

工具名称：add
工具描述：两个整数相加，返回结果。
参数 schema：{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## 2. Tool.from_function：手动创建工具

In [20]:
from langchain_core.tools import Tool, tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

# 方式二：使用 Tool.from_function 手动创建工具
# 适用于已有函数，不想用装饰器的场景

def get_weather(city: str) -> str:
    """根据城市名获取天气信息（模拟数据）。"""
    mock_weather = {
        "北京": "晴天，25°C",
        "上海": "多云，22°C",
        "广州": "小雨，28°C",
    }
    return mock_weather.get(city, f"未找到{city}的天气信息")

weather_tool = Tool.from_function(
    func=get_weather,
    name="get_weather",
    description="根据城市名称查询当前天气信息，输入为城市名字符串。",
)

@tool(description="根据城市名称。",parse_docstring=True,name_or_callable="getWeather1")
def get_weather1(city: str, time: str = "今天" ) -> str:
    """
    根据城市名获取天气信息（模拟数据）。
    
    Args:
        city (str): 城市名称。
        time (str, optional): 查询时间，默认为今天。
    
    Returns:
        str: 天气信息。
    """
    mock_weather = {
        "北京": "晴天，25°C",
        "上海": "多云，22°C",
        "广州": "小雨，28°C",
    }
    return mock_weather.get(city, f"未找到{city}的天气信息")

rprint(convert_to_openai_tool(get_weather1))

# 测试
result = weather_tool.invoke("北京")
print(f"天气查询结果：{result}")
print(f"\n工具名称：{weather_tool.name}")
print(f"工具描述：{weather_tool.description}")
print(f"参数 schema：{weather_tool.args}")

{
    'type': 'function',
    'function': {
        'name': 'getWeather1',
        'description': '根据城市名称。',
        'parameters': {
            'properties': {
                'city': {'description': '城市名称。', 'type': 'string'},
                'time': {'default': '今天', 'description': '查询时间，默认为今天。', 'type': 'string'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

天气查询结果：晴天，25°C

工具名称：get_weather
工具描述：根据城市名称查询当前天气信息，输入为城市名字符串。
参数 schema：{'tool_input': {'type': 'string'}}


## 3. LLM 工具调用：bind_tools 将工具绑定到 LLM

In [3]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

dotenv.load_dotenv()

# 创建工具
@tool
def calculate(expression: str) -> str:
    """计算数学表达式，输入为数学表达式字符串，如 '2 + 3 * 4'。"""
    try:
        result = eval(expression)
        return f"计算结果：{expression} = {result}"
    except Exception as e:
        return f"计算错误：{e}"

@tool
def get_weather(city: str) -> str:
    """根据城市名称查询当前天气信息。"""
    mock_weather = {
        "北京": "晴天，25°C",
        "上海": "多云，22°C",
    }
    return mock_weather.get(city, f"未找到{city}的天气信息")

# 初始化 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

# bind_tools：将工具绑定到 LLM，LLM 会知道有哪些工具可用
llm_with_tools = llm.bind_tools([calculate, get_weather])

# LLM 会自动决定是否调用工具
response = llm_with_tools.invoke("北京天气怎么样？")
print(f"LLM 响应：{response.content}")
print(f"工具调用：{response.tool_calls}")

# 再试一个需要计算的
response2 = llm_with_tools.invoke("请计算 (3 + 5) * 2")
print(f"\nLLM 响应：{response2.content}")
print(f"工具调用：{response2.tool_calls}")

LLM 响应：
工具调用：[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_d16d121b92a5483387a2ce43', 'type': 'tool_call'}]

LLM 响应：
工具调用：[{'name': 'calculate', 'args': {'expression': '(3 + 5) * 2'}, 'id': 'call_9304d4d3667046a080df61d5', 'type': 'tool_call'}]


## 4. 手动执行工具调用并返回结果给 LLM

In [6]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from rich import print as rprint

dotenv.load_dotenv()

# 工具定义
tools = {
    "calculate": lambda expr: str(eval(expr)),
    "get_weather": lambda city: {"北京": "晴天25°C", "上海": "多云22°C"}.get(city, "未知"),
}

# 初始化 LLM 并绑定工具
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

@tool
def calculate(expression: str) -> str:
    """计算数学表达式。"""
    return str(eval(expression))

@tool
def get_weather(city: str) -> str:
    """查询城市天气。"""
    return {"北京": "晴天25°C", "上海": "多云22°C"}.get(city, "未知")


rprint("工具定义：",calculate,get_weather)

llm_with_tools = llm.bind_tools([calculate, get_weather])

# 第一轮：发送问题
messages = [HumanMessage(content="帮我算一下 (10 + 20) * 3")]
response = llm_with_tools.invoke(messages)
print(f"LLM 返回的工具调用：{response.tool_calls}")

# 第二轮：手动执行工具，将结果追加到消息
messages.append(response)

for tool_call in response.tool_calls:
    # 根据工具名找到对应的工具并执行
    tool_map = {"calculate": calculate, "get_weather": get_weather}
    tool_result = tool_map[tool_call["name"]].invoke(tool_call["args"])
    print(f"工具 {tool_call['name']} 结果：{tool_result}")

    # 将工具执行结果作为 ToolMessage 追加到对话
    messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"]))

rprint("工具调用结果：",messages)
# 第三轮：将工具结果发回 LLM，让 LLM 生成最终回答
final_response = llm_with_tools.invoke(messages)
print(f"\n最终回答：{final_response.content}")

工具定义：
StructuredTool(
    name='calculate',
    description='计算数学表达式。',
    args_schema=<class 'langchain_core.utils.pydantic.calculate'>,
    func=<function calculate at 0x00000252738BFCE0>
)
StructuredTool(
    name='get_weather',
    description='查询城市天气。',
    args_schema=<class 'langchain_core.utils.pydantic.get_weather'>,
    func=<function get_weather at 0x0000025277322700>
)

LLM 返回的工具调用：[{'name': 'calculate', 'args': {'expression': '(10 + 20) * 3'}, 'id': 'call_80319eb959d041f2a1caf4e3', 'type': 'tool_call'}]
工具 calculate 结果：90


工具调用结果：
[
    HumanMessage(content='帮我算一下 (10 + 20) * 3', additional_kwargs={}, response_metadata={}),
    AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 76,
                'prompt_tokens': 561,
                'total_tokens': 637,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 44,
                    'rejected_prediction_tokens': None
                },
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512}
            },
            'model_provider': 'openai',
            'model_name': 'mimo-v2.5-pro',
            'system_fingerprint': None,
            'id': 'cbdb64d2e7fc41879a7e78d0bb5f00c5',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f1c95-9c85-76f2-b10a-bf49956a39b0-0',
        tool_calls=[
            {
                'name': 'calculate',
                'args': {'expression': '(10 + 20) * 3'},
                'id': 'call_80319eb959d041f2a1caf4e3',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 561,
            'output_tokens': 76,
            'total_tokens': 637,
            'input_token_details': {'cache_read': 512},
            'output_token_details': {'reasoning': 44}
        }
    ),
    ToolMessage(content='90', tool_call_id='call_80319eb959d041f2a1caf4e3')
]


最终回答：计算结果是 **90**。

运算过程：
1. 先算括号里的加法：10 + 20 = 30
2. 再乘以 3：30 × 3 = 90

如果你还有其他计算需要帮忙，随时告诉我哦！


## 5. 使用 Agent 自动调用工具（推荐）

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

dotenv.load_dotenv()

# 定义工具
@tool
def calculate(expression: str) -> str:
    """计算数学表达式，输入为数学表达式字符串。"""
    return str(eval(expression))

@tool
def get_weather(city: str) -> str:
    """根据城市名称查询当前天气信息。"""
    return {"北京": "晴天25°C", "上海": "多云22°C"}.get(city, "未知城市")

# 初始化 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

# 创建 prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的助手，可以使用工具来回答问题。"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 创建 agent
tools = [calculate, get_weather]
agent = create_tool_calling_agent(llm, tools, prompt)

# AgentExecutor 负责自动执行工具调用循环
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 运行
result = agent_executor.invoke({
    "chat_history": [],
    "input": "帮我算一下 (15 + 25) * 2 等于多少？然后查一下北京的天气。",
})
print(f"\n最终回答：{result['output']}")

## 6.自定义args_schema

In [31]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool


# 自定义 args_schema：使用 Pydantic 模型精确控制工具的参数校验
# 适用场景：需要详细的参数描述、类型校验、默认值、枚举值等

class SearchInput(BaseModel):
    """搜索引擎查询的输入参数。"""
    query: str = Field(description="搜索关键词")
    max_results: int = Field(default=5, ge=1, le=20, description="最大返回结果数，范围 1-20")
    # Pydantic V2 推荐用 Literal 替代 Field(enum=...)
    language: Literal["zh", "en"] = Field(default="zh", description="搜索语言，zh 中文 / en 英文")

@tool(args_schema=SearchInput)
def web_search(query: str, max_results: int = 5, language: str = "zh") -> str:
    """使用搜索引擎搜索互联网内容。"""
    # 模拟搜索结果
    return f"搜索「{query}」（{language}）返回 {max_results} 条结果：...\n结果1：...\n结果2：..."

rprint(f"工具定义：\n",convert_to_openai_tool(web_search))

# 测试
print(f"工具名称：{web_search.name}")
print(f"工具描述：{web_search.description}")
print(f"参数 schema：")
for name, prop in web_search.args.items():
    print(f"  {name}: {prop}")

print(f"\n调用结果：{web_search.invoke({'query': 'LangChain教程', 'max_results': 3})}")

# 验证：传入超出范围的 max_results 会报错
try:
    web_search.invoke({"query": "test", "max_results": 50})
except Exception as e:
    print(f"\n校验错误：{e}")

工具定义：

{
    'type': 'function',
    'function': {
        'name': 'web_search',
        'description': '使用搜索引擎搜索互联网内容。',
        'parameters': {
            'properties': {
                'query': {'description': '搜索关键词', 'type': 'string'},
                'max_results': {
                    'default': 5,
                    'description': '最大返回结果数，范围 1-20',
                    'maximum': 20,
                    'minimum': 1,
                    'type': 'integer'
                },
                'language': {
                    'default': 'zh',
                    'description': '搜索语言，zh 中文 / en 英文',
                    'enum': ['zh', 'en'],
                    'type': 'string'
                }
            },
            'required': ['query'],
            'type': 'object'
        }
    }
}

工具名称：web_search
工具描述：使用搜索引擎搜索互联网内容。
参数 schema：
  query: {'description': '搜索关键词', 'title': 'Query', 'type': 'string'}
  max_results: {'default': 5, 'description': '最大返回结果数，范围 1-20', 'maximum': 20, 'minimum': 1, 'title': 'Max Results', 'type': 'integer'}
  language: {'default': 'zh', 'description': '搜索语言，zh 中文 / en 英文', 'enum': ['zh', 'en'], 'title': 'Language', 'type': 'string'}

调用结果：搜索「LangChain教程」（zh）返回 3 条结果：...
结果1：...
结果2：...

校验错误：1 validation error for SearchInput
max_results
  Input should be less than or equal to 20 [type=less_than_equal, input_value=50, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
